##### file1.csv ：
前两列: id1 和 id2 用来标识一个样本。这两列的值如果相同，则表示这两个记录对应的是同一个样本。
从第三列开始到最后一列：这些列的值构成该样本的多标签分类的标签。
##### file2.csv ：
前两列: id1 和 id2 用来标识一个样本。这两列的值如果相同，则表示这两个记录对应的是同一个样本。
第三列: image 属性表示该样本对应的图像文件名。需要注意的是，一个样本可能有多张图像，所以在 file2.csv 中，如果同一个样本有多张图像，它会占据多行数据。
第四列：split 属性表示数据的分割方式，取值可以是 train、val 或 test ，分别对应训练集、验证集或者测试集。

### 目标

在 task.py 文件中，你需要实现 csv2json(file_path1: str, file_path2: str, file_path3: str)->None 方法。该方法的目标是将两个 CSV 文件中的数据整合到一个 JSON 文件中。具体的合并规则如下：

根据 file1.csv 和 file2.csv 中的 id1 和 id2 属性值相同的样本，将这些样本合并为一个完整的样本。合并后的样本应包括以下属性：

id1 和 id2 ：用于标识样本的唯一属性。
label ：表示该样本的多标签分类标签，值为一个列表。
images ：表示该样本所包含的图像名称，值为一个列表。
split ：表示该样本是训练数据（train）、验证数据（val）还是测试数据（test）
将合并后的数据根据 split 属性的值，将每个样本分别归类为 train（训练数据）、val（验证数据）和 test（测试数据）。

最终将上述合并并分类后的数据以 JSON 格式保存到指定的文件路径 file_path3 中，JSON 中的列表元素顺序应与文件内容顺序一致。

该方法参数的含义如下：
file_path1 : file1.csv 的文件路径。
file_path2 : file2.csv 的文件路径。
file_path3 : JSON 文件的保存路径。
该方法的返回值：无

In [8]:
def csv2json(file_path1: str, file_path2: str, file_path3: str)->None:
    
    import json
    import csv
    # 第一步：读取 file1.csv，提取 label
    # 格式：{(id1, id2): [label_list]}
    labels_dict = {}
    with open(file_path1, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)  # 跳过表头
        for row in reader:
            id1, id2 = row[0], row[1]
            # 剩下的 row[2:] 全是 label 的 0/1 标志，转为 int 列表
            label = [int(x) for x in row[2:]]
            labels_dict[(id1, id2)] = label
    # 第二步：读取 file2.csv，合并图像路径和 split 信息
    # 因为一个 (id1, id2) 对应多张图，所以需要合并
    merged_data = {}
    # 为了保证输出顺序与文件内容顺序一致，我们记录 key 出现的先后顺序
    order_keys = []

    with open(file_path2, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            key = (row['id1'], row['id2'])
            
            if key not in merged_data:
                order_keys.append(key)
                merged_data[key] = {
                    "id1": row['id1'],
                    "id2": row['id2'],
                    "label": labels_dict.get(key, []),
                    "images": [],
                    "split": row['split']
                }
            # 将当前行的图片名加入列表
            merged_data[key]["images"].append(row['image'])

    # 第三步：按照 split 归类
    result = {"train": [], "val": [], "test": []}
    
    for key in order_keys:
        item = merged_data[key]
        split_type = item["split"]
        if split_type in result:
            result[split_type].append(item)

    # 第四步：写入 JSON 文件
    with open(file_path3, 'w', encoding='utf-8') as f:
        # ensure_ascii=False 保证中文（如有）正常显示，indent=4 保持美观
        json.dump(result, f, indent=4)

    
if __name__ == '__main__':
    file_path1 = r'./file1.csv'
    file_path2 = r'./file2.csv'
    file_path3 = r'./file3.json'
    csv2json(file_path1, file_path2, file_path3)